In [ ]:
import csv
import math
import networkx as nx

In [ ]:
# Load data from csv
rows = []
with open("Desktop/extract_usernames/extract_usernames.csv", newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append((r["username"], r["templates"].split(";")))
 
# 3. Build the graph
G = nx.Graph()
story_ids = sorted({t for _, ts in rows for t in ts})
 
for t in story_ids:
    G.add_node(t, node_type="story")
 
for username, templates in rows:
    category = "recurring" if len(templates) > 1 else "unique"
    G.add_node(username, node_type="reposter", category=category)
    for t in templates:
        G.add_edge(username, t, weight=len(templates))
 
# Repost count and colors
repost_count = {
    "template_1": 47700, "template_2": 24400, "template_3": 45400, "template_4": 27000,
    "template_5": 386, "template_6": 15200, "template_7": 30100, "template_8": 796000,
    "template_9": 985000, "template_10": 28000, "template_11": 483000, "template_12": 16900,
    "template_13": 72900, "template_14": 42900,
}
 
story_color = dict(zip(story_ids, [
    (242, 220, 93), (242, 163, 89), (219, 144, 101), (161, 134, 158), (36, 11, 54),
    (224, 141, 172), (106, 127, 219), (87, 226, 229), (69, 203, 133), (21, 49, 49),
    (218, 239, 179), (234, 158, 141), (30, 85, 92), (182, 198, 73),
]))
 
# Node size and color
degree = dict(G.degree())
 
for n, d in G.nodes(data=True):
    if d["node_type"] == "story":
        size = round(8 + 20 * math.log10(repost_count[n]), 2)
        r, g, b = story_color[n]
    else:
        size = round(8 + 8 * math.sqrt(degree[n]), 2)
        r, g, b = (230, 57, 70) if d["category"] == "recurring" else (190, 190, 190)
    d["viz"] = {"color": {"r": r, "g": g, "b": b}, "size": size}
 
# Edge opacity and thickness based on weight
max_w = max(d["weight"] for _, _, d in G.edges(data=True))
 
for u, v, d in G.edges(data=True):
    alpha = round(0.15 + 0.85 * (d["weight"] - 1) / (max_w - 1), 2)
    thickness = round(0.3 + 0.2 * (d["weight"] - 1), 2)
    d["viz"] = {"color": {"r": 120, "g": 120, "b": 120, "a": alpha}, "thickness": thickness}
 
# Export GEXF file
nx.write_gexf(G, "reposter_network.gexf")